In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
os.listdir('/kaggle/input/competitions/home-credit-default-risk')


In [ ]:
# Install libraries and imports

!pip install shap xgboost lightgbm fairlearn -q

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.impute import SimpleImputer

import xgboost as xgb
import lightgbm as lgb
import shap
from fairlearn.metrics import (MetricFrame, selection_rate,
                                false_positive_rate, false_negative_rate,
                                demographic_parity_difference,
                                equalized_odds_difference)

import warnings
warnings.filterwarnings('ignore')

os.makedirs('images', exist_ok=True)

In [ ]:
# Load data

DATA_PATH = '/kaggle/input/competitions/home-credit-default-risk'

# Main application table
app_train = pd.read_csv(f'{DATA_PATH}/application_train.csv')

print(f"Shape: {app_train.shape}")
print(f"\nTarget distribution:")
print(app_train['TARGET'].value_counts(normalize=True))
print(f"\nDefault rate: {app_train['TARGET'].mean():.2%}")

In [ ]:
# Exploratory Data Analysis
# Target distribution

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
target_counts = app_train['TARGET'].value_counts()
axes[0].bar(['Repaid (0)', 'Defaulted (1)'], target_counts.values,
            color=['steelblue', 'coral'])
axes[0].set_title('Loan Repayment Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Applications')

for i, v in enumerate(target_counts.values):
    pct = v / len(app_train) * 100
    axes[0].text(i, v + 3000, f'{v:,}\n({pct:.1f}%)', ha='center', fontweight='bold')

# Default rate by contract type
default_by_contract = app_train.groupby('NAME_CONTRACT_TYPE')['TARGET'].mean()
axes[1].bar(default_by_contract.index, default_by_contract.values,
            color=['steelblue', 'coral'])
axes[1].set_title('Default Rate by Contract Type', fontweight='bold')
axes[1].set_ylabel('Default Rate')

plt.tight_layout()
plt.savefig('images/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n⚠️ Class imbalance detected — will need to handle this")

In [ ]:
# Feature correlation with target

# Get numerical features and their correlation with TARGET
numerical_cols = app_train.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('TARGET')

correlations = app_train[numerical_cols + ['TARGET']].corr()['TARGET'].sort_values(ascending=False)

# Plot top and bottom correlations
top_negative = correlations.tail(15)
top_positive = correlations.head(16).tail(15)  # skip TARGET itself

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_positive.plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Top 15 Positive Correlations with Default', fontweight='bold')
axes[0].set_xlabel('Correlation with TARGET')

top_negative.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Negative Correlations with Default', fontweight='bold')
axes[1].set_xlabel('Correlation with TARGET')

plt.tight_layout()
plt.savefig('images/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Preprocessing
# Handle missing values and encode categorical features

# Split features and target
y = app_train['TARGET']
X = app_train.drop(columns=['TARGET', 'SK_ID_CURR'])

# Identify column types
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features: {len(categorical_cols)}")
print(f"Numerical features: {len(numerical_cols)}")

# Encode categorical features
for col in categorical_cols:
    X[col] = X[col].fillna('Missing')
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Impute numerical features with median
num_imputer = SimpleImputer(strategy='median')
X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])

# Save sensitive attributes for fairness audit later
sensitive_attrs = X[['CODE_GENDER', 'DAYS_BIRTH']].copy()
sensitive_attrs['age_years'] = -sensitive_attrs['DAYS_BIRTH'] / 365
sensitive_attrs['age_group'] = pd.cut(sensitive_attrs['age_years'],
                                       bins=[0, 30, 45, 60, 100],
                                       labels=['Under 30', '30-45', '45-60', '60+'])

print(f"\nMissing values remaining: {X.isnull().sum().sum()}")

In [ ]:
# build a gender-free feature set

# Retain the original X for the fairness-labelled baseline, then drop the
# protected attribute for the compliant model.
X_nogender = X.drop(columns=['CODE_GENDER'])
print(f"Dropped CODE_GENDER. Feature count: {X.shape[1]} -> {X_nogender.shape[1]}")

In [ ]:
# Train/test split

X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
    X, y, sensitive_attrs, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%}, Test default rate: {y_test.mean():.2%}")

In [ ]:
# re-split (keep sensitive_attrs so the fairness audit is unchanged)

X_train_ng, X_test_ng, y_train_ng, y_test_ng, sensitive_train, sensitive_test = train_test_split(
    X_nogender, y, sensitive_attrs, test_size=0.2, random_state=42, stratify=y)
# random_state=42 + same y means these rows align exactly with your original split,
# so sensitive_test still matches y_test_ng row-for-row.

In [ ]:
# BASELINE MODELS
# Logistic Regression (regulatory-friendly baseline)

# Logistic regression is preferred by regulators because it's inherently interpretable
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
lr_model.fit(X_train_scaled, y_train)

lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
lr_ap = average_precision_score(y_test, lr_prob)
print(f"Logistic Regression — AUC: {lr_auc:.4f}, AP: {lr_ap:.4f}")

In [ ]:
# XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='auc', tree_method='hist')
xgb_model.fit(X_train, y_train)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)
xgb_ap = average_precision_score(y_test, xgb_prob)
print(f"XGBoost — AUC: {xgb_auc:.4f}, AP: {xgb_ap:.4f}")

In [ ]:
# LightGBM (often best for tabular data)

print("Training LightGBM...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    class_weight='balanced', random_state=42,
    verbose=-1)
lgb_model.fit(X_train, y_train)

lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_prob)
lgb_ap = average_precision_score(y_test, lgb_prob)
print(f"LightGBM — AUC: {lgb_auc:.4f}, AP: {lgb_ap:.4f}")

In [ ]:
#retrain the champion only (LightGBM)
# LightGBM retrained

lgb_model_ng = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    class_weight='balanced', random_state=42, verbose=-1)
lgb_model_ng.fit(X_train_ng, y_train_ng)

lgb_prob_ng = lgb_model_ng.predict_proba(X_test_ng)[:, 1]
lgb_auc_ng = roc_auc_score(y_test_ng, lgb_prob_ng)
lgb_ap_ng  = average_precision_score(y_test_ng, lgb_prob_ng)

print(f"LightGBM (no gender) — AUC: {lgb_auc_ng:.4f}, AP: {lgb_ap_ng:.4f}")
print(f"AUC delta vs original: {lgb_auc_ng - 0.7613:+.4f}")
print(f"AP  delta vs original: {lgb_ap_ng - 0.2509:+.4f}")

In [ ]:
# compare models

print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"{'Model':<25} {'AUC-ROC':>10} {'AP':>10}")
print("-"*60)
print(f"{'Logistic Regression':<25} {lr_auc:>10.4f} {lr_ap:>10.4f}")
print(f"{'XGBoost':<25} {xgb_auc:>10.4f} {xgb_ap:>10.4f}")
print(f"{'LightGBM':<25} {lgb_auc:>10.4f} {lgb_ap:>10.4f}")
print("="*60)

# ROC curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, prob, auc, color in [
    ('Logistic Regression', lr_prob, lr_auc, 'steelblue'),
    ('XGBoost', xgb_prob, xgb_auc, 'orange'),
    ('LightGBM', lgb_prob, lgb_auc, 'coral')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR curve
for name, prob, ap, color in [
    ('Logistic Regression', lr_prob, lr_ap, 'steelblue'),
    ('XGBoost', xgb_prob, xgb_ap, 'orange'),
    ('LightGBM', lgb_prob, lgb_ap, 'coral')]:
    precision, recall, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(recall, precision, color=color, linewidth=2, label=f'{name} (AP = {ap:.3f})')

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.5,
                label=f'Random ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve Comparison', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pick best model and confusion matrix

# Pick LightGBM as champion (typically best on tabular data)
best_model = lgb_model
best_prob = lgb_prob
best_pred = (best_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Repay', 'Predicted Default'],
            yticklabels=['Actual Repay', 'Actual Default'])
ax.set_title('Confusion Matrix — LightGBM (threshold=0.5)', fontweight='bold')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, best_pred, target_names=['Repay', 'Default']))

In [ ]:
# SHAP Explainability
# SHAP summary plot (feature importance)

X_sample = X_test.sample(5000, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
plt.title('Top 20 Features Driving Default Predictions (SHAP)', fontweight='bold')
plt.tight_layout()
plt.savefig('images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP Summary — Feature Values vs. Default Impact', fontweight='bold')
plt.tight_layout()
plt.savefig('images/shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP dependence plots for top 4 features

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features_idx = np.argsort(mean_abs_shap)[-4:][::-1]
top_features = X_sample.columns[top_features_idx].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feature in zip(axes.flatten(), top_features):
    shap.dependence_plot(feature, shap_values, X_sample, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feature}', fontweight='bold')

plt.tight_layout()
plt.savefig('images/shap_dependence_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Rebuild the SHAP chain on the no-gender model
best_model = lgb_model_ng
X_sample = X_test_ng.sample(5000, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    ev = np.array(expected_value)
    expected_value = expected_value[1] if ev.ndim > 0 and len(ev) > 1 else expected_value

In [ ]:
# Individual prediction explanations (adverse action notices)

X_sample_sorted = X_sample.reset_index(drop=True)
sample_probs = best_model.predict_proba(X_sample_sorted)[:, 1]

high_risk_positions = np.argsort(sample_probs)[-3:][::-1]

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1] if len(np.array(expected_value).shape) > 0 and len(expected_value) > 1 else expected_value

for i, pos in enumerate(high_risk_positions):
    fig, ax = plt.subplots(figsize=(16, 8))
    plt.sca(ax)

    shap.plots._waterfall.waterfall_legacy(
        expected_value,
        shap_values[pos],
        X_sample_sorted.iloc[pos],
        max_display=10,
        show=False
    )

    ax.set_title(
        f'High-Risk Applicant #{i+1} (Predicted Default Probability: {sample_probs[pos]:.2%})',
        fontweight='bold', fontsize=13, pad=20
    )

    plt.subplots_adjust(left=0.35)

    plt.savefig(
        f'images/shap_individual_applicant_{i+1}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()

print("\n" + "="*70)
print("ADVERSE ACTION NOTICES (top 3 high-risk applicants)")
print("="*70)

for rank, pos in enumerate(high_risk_positions):
    prediction = sample_probs[pos]
    contributions = pd.Series(shap_values[pos], index=X_sample_sorted.columns)
    top_negative = contributions.nlargest(5)

    print(f"\nApplicant #{rank+1} (predicted default probability: {prediction:.2%})")
    print("Top reasons for decline:")
    for feature, contribution in top_negative.items():
        value = X_sample_sorted.iloc[pos][feature]
        print(f"  - {feature}: {value:.2f} (SHAP contribution: +{contribution:.4f})")
    print("-" * 70)

In [ ]:
# Get the actual row indices of the top 3 high-risk applicants
top_3_positions = high_risk_positions

print("Verifying applicants are genuinely different:\n")

for rank, pos in enumerate(top_3_positions):
    row = X_sample_sorted.iloc[pos]
    prob = sample_probs[pos]
    print(f"Applicant #{rank+1} — Predicted probability: {prob:.6f}")
    print(f"  Position in sample: {pos}")
    # Show a few distinguishing features to confirm they are different people
    for feature in ['DAYS_BIRTH', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
        if feature in row.index:
            print(f"  {feature}: {row[feature]:.2f}")
    print("-" * 60)

In [ ]:
# Fairness by gender (excluding XNA) and age

sensitive_test_labels = sensitive_test.copy()
sensitive_test_labels['gender_label'] = sensitive_test_labels['CODE_GENDER'].map({0: 'F', 1: 'M', 2: 'XNA'})

# Exclude XNA (n≈4, statistically meaningless subgroup)
mf_mask = sensitive_test_labels['gender_label'].isin(['M', 'F'])
y_test_mf = y_test[mf_mask]
best_pred_mf = best_pred[mf_mask]
gender_mf = sensitive_test_labels.loc[mf_mask, 'gender_label']

gender_metrics = MetricFrame(
    metrics={
        'selection_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate,
    },
    y_true=y_test_mf,
    y_pred=best_pred_mf,
    sensitive_features=gender_mf
)

print("Fairness Metrics by Gender (M vs F, XNA excluded):")
print(gender_metrics.by_group)
print(f"\nDemographic Parity Difference: {demographic_parity_difference(y_test_mf, best_pred_mf, sensitive_features=gender_mf):.4f}")
print(f"Equalised Odds Difference: {equalized_odds_difference(y_test_mf, best_pred_mf, sensitive_features=gender_mf):.4f}")

age_metrics = MetricFrame(
    metrics={
        'selection_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate,
    },
    y_true=y_test,
    y_pred=best_pred,
    sensitive_features=sensitive_test_labels['age_group']
)

print("\nFairness Metrics by Age Group:")
print(age_metrics.by_group)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

gender_metrics.by_group['selection_rate'].plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Decline Rate by Gender', fontweight='bold')
axes[0, 0].set_ylabel('Selection Rate (Decline)')

gender_metrics.by_group['false_positive_rate'].plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('False Positive Rate by Gender', fontweight='bold')

age_metrics.by_group['selection_rate'].plot(kind='bar', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('Decline Rate by Age Group', fontweight='bold')

age_metrics.by_group['false_positive_rate'].plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_title('False Positive Rate by Age Group', fontweight='bold')

plt.tight_layout()
plt.savefig('images/fairness_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# re-run fairness on the gender-free model

best_pred_ng = (lgb_prob_ng >= 0.5).astype(int)

sensitive_test_labels = sensitive_test.copy()
# Verify the encoding rather than trusting {0:'F',1:'M',2:'XNA'} — LabelEncoder
# assigns codes alphabetically, so confirm before mapping.
print(sensitive_test_labels['CODE_GENDER'].value_counts())
sensitive_test_labels['gender_label'] = sensitive_test_labels['CODE_GENDER'].map(
    {0: 'F', 1: 'M', 2: 'XNA'})

mf_mask = sensitive_test_labels['gender_label'].isin(['M', 'F'])
y_mf     = y_test_ng[mf_mask]
pred_mf  = best_pred_ng[mf_mask]
gender_mf = sensitive_test_labels.loc[mf_mask, 'gender_label']

dpd_ng = demographic_parity_difference(y_mf, pred_mf, sensitive_features=gender_mf)
eod_ng = equalized_odds_difference(y_mf, pred_mf, sensitive_features=gender_mf)

print(f"\nDemographic Parity Difference (no gender): {dpd_ng:.4f}  (was 0.1586)")
print(f"Equalised Odds Difference (no gender):     {eod_ng:.4f}  (was 0.1617)")

gender_metrics_ng = MetricFrame(
    metrics={'selection_rate': selection_rate,
             'false_positive_rate': false_positive_rate,
             'false_negative_rate': false_negative_rate},
    y_true=y_mf, y_pred=pred_mf, sensitive_features=gender_mf)
print(gender_metrics_ng.by_group)